In [ ]:
import pandas as pd

books_final = pd.read_csv('data/books_final.csv')
print(books_final.shape)
books_final.head()

In [36]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = modelo.encode(
    books_final['genero_texto'].tolist(),
    show_progress_bar=True
)

print(embeddings.shape)

c:\Users\jvict\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\jvict\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jvict\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to ru

(9999, 384)


In [37]:
import numpy as np
np.save('data/embeddings.npy', embeddings)

In [38]:
from sklearn.metrics.pairwise import cosine_similarity

similaridade = cosine_similarity(embeddings)
print(similaridade.shape)

(9999, 9999)


In [39]:
indices = pd.Series(books_final.index, index=books_final['title']).drop_duplicates()

In [43]:
def recommender(titulo, n=5):
    if titulo not in indices:
        print(f"Livro '{titulo}' não encontrado na base.")
        return None
    
    idx = indices[titulo]
    
    # pega as similaridades desse livro com todos os outros
    scores = list(enumerate(similaridade[idx]))
    
    # ordena do mais parecido pro menos parecido
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    # pega os top N, ignorando o próprio livro (posição 0)
    scores = scores[1:n+1]
    
    livros_recomendados = books_final.iloc[[i[0] for i in scores]][['title', 'authors', 'genero_texto', 'average_rating']].copy()
    livros_recomendados['similaridade'] = [i[1] for i in scores]
    
    return livros_recomendados

In [47]:
recommender('Pride and Prejudice')

,title,authors,genero_texto,average_rating,similaridade
171,Anna Karenina,"Leo Tolstoy, Louise Maude, Leo Tolstoj, Aylmer...",classics fiction romance historical-fiction hi...,4.02,1.0
1112,North and South,"Elizabeth Gaskell, Alan Shelston",classics fiction romance historical-fiction hi...,4.13,1.0
3457,Les Liaisons dangereuses,"Pierre-Ambroise Choderlos de Laclos, Douglas P...",classics fiction romance historical-fiction hi...,4.07,1.0
4744,La Dame aux Camélias,"Alexandre Dumas fils, David Coward",classics fiction romance historical-fiction hi...,3.97,1.0
4868,Shirley,"Charlotte Brontë, Lucasta Miller, collaborativ...",classics fiction romance historical-fiction hi...,3.71,1.0
